# Lekcija 10 - AI agenti v produkciji

V tej lekciji boste spoznali **produkcijske vzorce** za AI agente z uporabo Microsoft Agent Framework (Python). Obravnavamo:

- **Opazovanje** — dodajanje merjenja časa in beleženja interakcij z agentom
- **Vrednotenje** — uporaba agenta ocenjevalca za oceno kakovosti odgovorov
- **Upravljanje stroškov** — strategije za optimizacijo tokenov in izbiro modela

Scenarij je **potovalni agent**, ki uporabnikom pomaga načrtovati potovanja, z nadzorom in vrednotenjem na vrhu.


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Premisleki za proizvodnjo

Premestitev AI agentov iz prototipov v proizvodnjo zahteva temeljito pozornost trem stebrom:

1. **Opazovanje** — Potrebujete vpogled v to, kaj agent počne, koliko časa traja in katere orodja pokliče. Brez sledenja in beleženja je odpravljanje težav v proizvodnji skoraj nemogoče.

2. **Vrednotenje** — Avtomatizirane kontrole kakovosti zagotavljajo, da agentovi odgovori ostajajo natančni, popolni in koristni skozi čas. Evaluacijski agent lahko ovrednoti odgovore glede na določena merila.

3. **Upravljanje stroškov** — Uporaba tokenov neposredno vpliva na stroške. Strategije, kot so optimizacija pozivov, izbira modela in predpomnjenje, pomagajo obvladovati stroške brez žrtvovanja kakovosti.


## Gradnja opazovalnega agenta

Opredelimo orodja za potovanje in ovijemo klic agenta s časovnim merjenjem, da lahko spremljamo latenco. V produkciji bi integrirali z OpenTelemetry ali podobnim sledilnim sistemom.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzorci ocenjevanja

Pogost proizvodni vzorec je uporaba drugega agenta kot **ocenjevalca**. Ocenjevalec oceni odziv primarnega agenta glede na vnaprej določena merila, kot so popolnost, natančnost in koristnost.

To omogoča:
- Avtomatizirane kakovostne pregrade, preden odzivi dosežejo uporabnike
- Zaznavanje regresije, ko se spremenijo pozivi ali modeli
- Neprestano spremljanje uspešnosti agenta skozi čas


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategije upravljanja stroškov

Nadzor stroškov je ključen za produkcijske AI agente. Tukaj so ključne strategije:

| Strategija | Opis |
|---|---|
| **Optimizacija poziva** | Ohranjajte sistemska navodila jedrnata. Odstranite odvečen kontekst za zmanjšanje vhodnih tokenov. |
    "| **Izbira modela** | Uporabite manjše, cenejše modele (npr. GPT-4.1-mini) za enostavne naloge, kot so klasifikacija ali ekstrakcija, in rezervirajte večje modele za kompleksno sklepanje. |\n",
| **Predpomnjenje** | Predpomnite rezultate orodij in pogoste poizvedbe, da se izognete odvečnim API klicem. |
| **Proračuni tokenov** | Nastavite omejitve `max_tokens`, da preprečite nepričakovano dolge odgovore. |
| **Serijsko obdelovanje** | Združite več uporabniških poizvedb v en sam API klic, kjer je mogoče. |

V praksi se dobro obnese večstopenjski pristop: preprosta povpraševanja usmerite na hiter, poceni model in zahtevnejša poizvedovanja napotite na zmogljivejšega.


## Povzetek

V tej lekciji ste se naučili, kako:

1. **Dodati opazovanje** interakcijam agenta z merjenjem časa in beleženjem, kar postavlja temelje za sledenje in spremljanje.
2. **Samodejno oceniti odzive agenta** z uporabo ocenjevalnega agenta, ki ocenjuje popolnost, natančnost in koristnost.
3. **Upravljati stroške** z optimizacijo pozivov, izborom modela, predpomnjenjem in proračuni za tokene.

Ti proizvodni vzorci pomagajo zagotoviti, da so vaši AI agenti zanesljivi, merljivi in stroškovno učinkoviti v velikem obsegu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
